# Quantum Kernel Evaluation Pipeline

This notebook constructs the meta-dataset for the encoding circuit recommendation framework. It executes three stages:
1. Load 200 binary classification datasets (174 synthetic + 26 real-world)
2. Evaluate 9 encoding circuits × 3 classifiers (SVC, GPC, KRC) × 10 random splits per dataset
3. Compute 24 classical complexity metrics per dataset

Outputs: `taska_labels.csv`, `taskb_labels.csv`, `synthesis_df.csv` — inputs for the Majority Voting and LOOCV recommendation steps.

## 1. Setup

In [16]:
from Qsun.Qkernels import *
from Qsun.Qgates import *
from Qsun.Qmeas import *
from Qsun.Qcircuit import *
from Qsun.Qwave import *
from Qsun.Qencodes import *
from Qsun.Qdata import *

import numpy as np
import matplotlib.pyplot as plt
from datasets.load_data import *
from src.kernel_evaluation import *
from src.config import *

import problexity as px

np.random.seed(1234)

## 2. Kernel Evaluation Routines

`total_runs` evaluates all 9 encoding circuits × 3 classifiers on a single dataset, repeated over 10 random train/test splits. Each run applies PCA (if the dataset exceeds 4 features) and MinMax scaling to [0, π] before computing quantum kernel matrices via state fidelity.

`run_all_datasets` iterates `total_runs` across the full dataset collection.

In [17]:
def total_runs(dataset_name="Iris",
               raw_datasets=None,     
               encodings=None,
               n_layers=2,
               n_runs=10,
               test_size=0.2,
               random_state=42,
               feature_range=(0, np.pi),
               max_qubit=4):

    if encodings is None:
        encodings = get_available_encodings()

    print(f"Dataset: {dataset_name}")

    results_accumulator = {
        enc: {m: {"train": [], "test": []} for m in ["SVM", "GPC", "KRC"]}
        for enc in encodings
    }

    X_full, y_full = raw_datasets[dataset_name]

    for run in range(n_runs):
        seed = random_state + run

        X_train, X_test, y_train, y_test = train_test_split(
            X_full, y_full,
            test_size=test_size,
            random_state=seed,
            stratify=y_full
        )

        if X_train.shape[1] > max_qubit:
            pca = PCA(n_components=max_qubit, random_state=seed)
            X_train = pca.fit_transform(X_train)
            X_test = pca.transform(X_test)

        scaler = MinMaxScaler(feature_range=feature_range)
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        kernels = total_kernels(
            X_train, X_test,
            encoding_names=encodings,
            n_layers=n_layers,
            random_state=seed,
            verbose=False
        )

        for enc_name, (K_train, K_test) in kernels.items():
            for model_name in ["SVM", "GPC", "KRC"]:
                result = evaluate_kernel(
                    K_train, K_test, y_train, y_test,
                    enc_name, model_name
                )
                results_accumulator[enc_name][model_name]["train"].append(result.train_accuracy)
                results_accumulator[enc_name][model_name]["test"].append(result.test_accuracy)

        print(f"  Run {run+1}/{n_runs} completed", end='\r')

    print()

    all_results = {}
    for enc_name in encodings:
        enc_results = []
        for model_name in ["SVM", "GPC", "KRC"]:
            train_scores = results_accumulator[enc_name][model_name]["train"]
            test_scores = results_accumulator[enc_name][model_name]["test"]

            enc_results.append(KernelEvaluation(
                model_name=model_name,
                encoding_name=enc_name,
                train_accuracy=np.mean(train_scores),
                test_accuracy=np.mean(test_scores),
                train_std=np.std(train_scores),
                test_std=np.std(test_scores)
            ))
        all_results[enc_name] = enc_results

    return {"results": all_results}


def run_all_datasets(encodings=None, 
                     n_layers=2, 
                     n_runs=10, 
                     test_size=0.2, 
                     random_state=42, 
                     include_variants=True):

    datasets = load_extended_datasets(max_qubit=4, verbose=True)
   
    all_results = {}
    dataset_names = list(datasets.keys())
    
    print(f"Total datasets to process: {len(dataset_names)}")
    print("=" * 60)
    
    for idx, dataset_name in enumerate(dataset_names, 1):
        print(f"\n[{idx}/{len(dataset_names)}] Processing {dataset_name}...")
        
        result = total_runs(
            dataset_name=dataset_name,
            encodings=encodings,
            n_layers=n_layers,
            n_runs=n_runs,
            test_size=test_size,
            random_state=random_state,
            include_variants=include_variants
        )
        
        all_results[dataset_name] = result
    
    print("\n" + "=" * 60)
    print("All datasets processed!")
    
    return all_results

## 3. Summary Utilities

- `summary`: prints the encoding circuit × classifier accuracy table for a single dataset.
- `create_summary_table`: builds a cross-dataset DataFrame for one classifier, used for paper comparison tables.

In [18]:
def summary(all_results):
    print(f"{'Encoding':<22} {'Model':<6} {'Train':<18} {'Test':<18}")
    print("-" * 75)
    
    best_test_acc = 0
    best_config = None
    
    for encoding_name, results in all_results.items():
        for r in results: 
            train_str = f"{r.train_accuracy:.4f} ± {r.train_std:.4f}"
            test_str = f"{r.test_accuracy:.4f} ± {r.test_std:.4f}"
            print(f"{r.encoding_name:<22} {r.model_name:<6} {train_str:<18} {test_str:<18}")
            
            if r.test_accuracy > best_test_acc:
                best_test_acc = r.test_accuracy
                best_config = r
    
    print("-" * 75)
    if best_config:
        print(f"Best: {best_config.encoding_name} + {best_config.model_name} "
              f"= {best_test_acc:.4f} ± {best_config.test_std:.4f}")

In [19]:
def create_summary_table(all_dataset_results, 
                         model_name="SVM",
                         show_std=False) -> pd.DataFrame:
    encodings = list(ENCODING_REGISTER.keys())
    
    table_data = []
    for dataset_name, result_dict in all_dataset_results.items():
        row = {"Dataset": dataset_name}
        results = result_dict["results"]
        
        for enc_name in encodings:
            if enc_name in results:
                for r in results[enc_name]:
                    if r.model_name == model_name:
                        if show_std:
                            row[enc_name] = f"{r.test_accuracy:.4f} ± {r.test_std:.4f}"
                        else:
                            row[enc_name] = r.test_accuracy
                        break
            else:
                row[enc_name] = None if not show_std else "-"
        
        table_data.append(row)
    
    df = pd.DataFrame(table_data)
    df = df.set_index("Dataset")
    
    return df

## 4. Main Execution

Run the full evaluation pipeline across all 200 datasets. Raw `(X, y)` pairs are loaded via `ExtendedDatasetLoader`; preprocessing (PCA + scaling) is applied per-run inside `total_runs` to ensure split-specific normalization.

Results are serialized to a pickle file to avoid re-running the multi-hour evaluation.

In [ ]:
loader = ExtendedDatasetLoader(random_state=42)
raw_datasets = loader.load_all_extended(verbose=True)

print(f"Total datasets: {len(raw_datasets)}")
print("=" * 60)

all_results = {}

for dataset_name in raw_datasets.keys():
    result = total_runs(
        dataset_name=dataset_name,
        raw_datasets=raw_datasets,
        n_runs=10,
        random_state=42,
        max_qubit=4,
    )
    all_results[dataset_name] = result

print(f"\n Completely Loading datasets")

In [ ]:
import pickle

with open('all_results with load_data.pkl', 'wb') as f:
    pickle.dump(all_results, f)
print(f"Saved {len(all_results)} results to all_results with load_data.pkl")

Saved 200 results to all_results with load_data.pkl


## 5. Result Summaries

Three views of the evaluation results:
1. Best encoding circuit per classifier per dataset
2. Best overall combination (classifier + encoding circuit) per dataset
3. Side-by-side classifier comparison per dataset

In [ ]:
print("\n" + "="*80)
print("SUMMARY: BEST KERNEL FOR EACH MODEL (ACROSS ALL DATASETS)")
print("="*80)

for model_name in ["SVM", "GPC", "KRC"]:
    print(f"\n{'='*70}")
    print(f"MODEL: {model_name}")
    print(f"{'='*70}")
    print(f"{'Dataset':<25} {'Best Kernel':<22} {'Test Accuracy':<18}")
    print("-" * 70)
    
    for dataset_name, result_dict in all_results.items():
        results = result_dict["results"]
        
        best_acc = 0
        best_kernel = None
        best_std = 0
        
        for enc_name, enc_results in results.items():
            for r in enc_results:
                if r.model_name == model_name and r.test_accuracy > best_acc:
                    best_acc = r.test_accuracy
                    best_kernel = enc_name
                    best_std = r.test_std
        
        acc_str = f"{best_acc:.4f} ± {best_std:.4f}"
        print(f"{dataset_name:<25} {best_kernel:<22} {acc_str:<18}")
    
    print("-" * 70)


SUMMARY: BEST KERNEL FOR EACH MODEL (ACROSS ALL DATASETS)

MODEL: SVM
Dataset                   Best Kernel            Test Accuracy     
----------------------------------------------------------------------
Blobs_F2C2_S100           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C2_S150           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C2_S200           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C3_S100           HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S150           HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S200           HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S100           HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S150           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C4_S200           HighDim                1.0000 ± 0.0000   
Blobs_F3C2_S100           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F3C2_S150           HZY_CZ                 1.0000 ± 0.0000   
Blobs_F3C2_S200           HighDim         

In [ ]:
print("\n" + "="*80)
print("SUMMARY: BEST MODEL + KERNEL FOR EACH DATASET")
print("="*80)
print(f"{'Dataset':<25} {'Best Model':<8} {'Best Kernel':<22} {'Test Accuracy':<18}")
print("-" * 80)

for dataset_name, result_dict in all_results.items():
    results = result_dict["results"]
    
    best_acc = 0
    best_kernel = None
    best_model = None
    best_std = 0
    
    for enc_name, enc_results in results.items():
        for r in enc_results:
            if r.test_accuracy > best_acc:
                best_acc = r.test_accuracy
                best_kernel = enc_name
                best_model = r.model_name
                best_std = r.test_std
    
    acc_str = f"{best_acc:.4f} ± {best_std:.4f}"
    print(f"{dataset_name:<25} {best_model:<8} {best_kernel:<22} {acc_str:<18}")

print("-" * 80)


SUMMARY: BEST MODEL + KERNEL FOR EACH DATASET
Dataset                   Best Model Best Kernel            Test Accuracy     
--------------------------------------------------------------------------------
Blobs_F2C2_S100           SVM      HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C2_S150           SVM      HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C2_S200           SVM      HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C3_S100           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S150           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C3_S200           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S100           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F2C4_S150           SVM      HZY_CZ                 1.0000 ± 0.0000   
Blobs_F2C4_S200           SVM      HighDim                1.0000 ± 0.0000   
Blobs_F3C2_S100           SVM      HZY_CZ                 1.0000 ± 0.0000   
Blobs_F3C2_S150        

In [ ]:
print("\n" + "="*80)
print("SUMMARY: MODEL COMPARISON (SIDE-BY-SIDE)")
print("="*80)

print(f"{'Dataset':<20} ", end="")
for model in ["SVM", "GPC", "KRC"]:
    print(f"{model:<25} ", end="")
print()
print("-" * 95)

for dataset_name, result_dict in all_results.items():
    results = result_dict["results"]
    print(f"{dataset_name:<20} ", end="")
    
    for model_name in ["SVM", "GPC", "KRC"]:
        best_acc = 0
        best_kernel = None
        best_std = 0
        
        for enc_name, enc_results in results.items():
            for r in enc_results:
                if r.model_name == model_name and r.test_accuracy > best_acc:
                    best_acc = r.test_accuracy
                    best_kernel = enc_name
                    best_std = r.test_std
        
        acc_str = f"{best_acc:.4f}±{best_std:.4f}"
        print(f"{acc_str:<25} ", end="")
    
    print()


SUMMARY: MODEL COMPARISON (SIDE-BY-SIDE)
Dataset              SVM                       GPC                       KRC                       
-----------------------------------------------------------------------------------------------
Blobs_F2C2_S100      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C2_S150      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C2_S200      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C3_S100      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C3_S150      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C3_S200      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C4_S100      1.0000±0.0000             1.0000±0.0000             1.0000±0.0000             
Blobs_F2C4_S150      1.0000±0.0000             1.0000±0.0000  

## 6. Ground Truth Label Generation

Generate labels for both task formulations from the evaluation results:

- **Task-A** (single-best): each dataset maps to the single encoding circuit achieving the highest accuracy, paired with the best classifier.
- **Task-B** (tolerance-based): each dataset maps to all encoding circuits within a 1% tolerance of the best accuracy, restricted to the best classifier only (no cross-classifier ties).

Restricting Task-B ties to the best classifier avoids inflating the label set with spurious matches from weaker classifiers.

In [ ]:
def label_dataframe_best_model_per_dataset(all_dataset_results, threshold=0.01):

    encodings = list(ENCODING_REGISTER.keys())
    models = ["SVM", "GPC", "KRC"]
    
    table_data = []
    tied_data = []
    
    for dataset_name, result_dict in all_dataset_results.items():
        results = result_dict["results"]
        
        best_acc = 0
        best_kernel = None
        best_model = None
        
        for enc_name in encodings:
            if enc_name in results:
                for r in results[enc_name]:
                    if r.test_accuracy > best_acc:
                        best_acc = r.test_accuracy
                        best_kernel = enc_name
                        best_model = r.model_name
        
        if best_acc == 0:
            continue
        
        table_data.append({
            "Dataset": dataset_name,
            "Best_Model": best_model,
            "Best_Kernel": best_kernel,
            "Accuracy": best_acc
        })
        
        if threshold is not None:
            for enc_name in encodings:
                if enc_name in results:
                    for r in results[enc_name]:
                        if r.model_name == best_model and r.test_accuracy >= best_acc - threshold:
                            tied_data.append({
                                "Dataset": dataset_name,
                                "Best_Model": best_model,  
                                "Best_Kernel": enc_name,
                                "Accuracy": r.test_accuracy
                            })
    
    df_best = pd.DataFrame(table_data).set_index("Dataset")
    
    if threshold is not None and tied_data:
        df_tied = pd.DataFrame(tied_data).set_index("Dataset")
        
        print("Model Distribution:")
        print("  Task-A (Best only):")
        model_counts = df_best["Best_Model"].value_counts()
        for model, count in model_counts.items():
            print(f"    {model}: {count} datasets ({count/len(df_best)*100:.1f}%)")
        
        print("\n  Task-B (Tied within best model):")
        model_counts_tied = df_tied["Best_Model"].value_counts()
        for model, count in model_counts_tied.items():
            print(f"    {model}: {count} samples ({count/len(df_tied)*100:.1f}%)")
        
        tied_per_dataset = df_tied.groupby(level=0).size()
        print(f"\n  Average tied kernels per dataset: {tied_per_dataset.mean():.2f}")
        print(f"  Min: {tied_per_dataset.min()}, Max: {tied_per_dataset.max()}")
        print()
        
        return df_best, df_tied
    
    return df_best

df_best, df_tied = label_dataframe_best_model_per_dataset(all_results, threshold=0.01)

print("="*70)
print("Task-A: Single BEST (Model + Kernel) per Dataset")
print("="*70)
print(df_best.head(10))
print(f"\nTotal samples: {len(df_best)}")

print("\n" + "="*70)
print("Task-B: Tied Kernels (within best model only)")
print("="*70)
print(df_tied.head(15))
print(f"\nTotal samples: {len(df_tied)}")

Model Distribution:
  Task 1-A (Best only):
    SVM: 113 datasets (56.5%)
    KRC: 62 datasets (31.0%)
    GPC: 25 datasets (12.5%)

  Task 1-B (Tied within best model):
    SVM: 324 samples (73.3%)
    KRC: 82 samples (18.6%)
    GPC: 36 samples (8.1%)

  Average tied kernels per dataset: 2.21
  Min: 1, Max: 6

Task-A: Single BEST (Model + Kernel) per Dataset
                Best_Model Best_Kernel  Accuracy
Dataset                                         
Blobs_F2C2_S100        SVM      HZY_CZ       1.0
Blobs_F2C2_S150        SVM      HZY_CZ       1.0
Blobs_F2C2_S200        SVM      HZY_CZ       1.0
Blobs_F2C3_S100        SVM     HighDim       1.0
Blobs_F2C3_S150        SVM     HighDim       1.0
Blobs_F2C3_S200        SVM     HighDim       1.0
Blobs_F2C4_S100        SVM     HighDim       1.0
Blobs_F2C4_S150        SVM      HZY_CZ       1.0
Blobs_F2C4_S200        SVM     HighDim       1.0
Blobs_F3C2_S100        SVM      HZY_CZ       1.0

Total samples: 200

Task-B: Tied Kernels (within

## 7. Complexity Metric Computation

Compute 24 classical complexity metrics per dataset:
- 22 metrics from Problexity (feature overlap, linearity, neighborhood, network, dimensionality, class imbalance)
- 2 metrics from Qsun in Qdata.py: Kolmogorov complexity `C(D)` and intrinsic dimensionality `dim_eff`

Metrics are computed on the training portion (scaled to [0, 1]) and averaged over 10 runs to reduce variance from stochastic metric estimators.

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [ ]:
complexities_datasets = {}

for name, (X, y) in raw_datasets.items():
    complexities_train = []
    
    for i in range(10):
        seed = 42 + i
    
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y,
            test_size=0.2,
            random_state=seed,
            stratify=y
        )
        
        scaler = MinMaxScaler(feature_range=(0, 1))
        X_tr = scaler.fit_transform(X_tr)
        
        cc = px.ComplexityCalculator()
        cc.fit(X_tr, y_tr)
        results = list(cc.complexity)
        results.append(kolmogorov_complex(X_tr)['best_bytes'])
        results.append(intrinsic_dim_from_cov(X_tr))
        
        complexities_train.append(results)
    
    complexities_train = np.array(complexities_train)
    complexities_datasets[name] = np.mean(complexities_train, axis=0)


In [ ]:
labels = cc._metrics() + ['kolmogorov', 'intrinsic']
print(f"Number of metrics: {len(labels)}")
print(f"Labels: {labels}")

df = pd.DataFrame.from_dict(complexities_datasets, orient='index', columns=labels)
df

Number of metrics: 24
Labels: ['f1', 'f1v', 'f2', 'f3', 'f4', 'l1', 'l2', 'l3', 'n1', 'n2', 'n3', 'n4', 't1', 'lsc', 'density', 'clsCoef', 'hubs', 't2', 't3', 't4', 'c1', 'c2', 'kolmogorov', 'intrinsic']


,f1,f1v,f2,f3,f4,l1,l2,l3,n1,n2,...,density,clsCoef,hubs,t2,t3,t4,c1,c2,kolmogorov,intrinsic
Blobs_F2C2_S100,0.007097,0.002049,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.006250,0.016410,...,0.508924,0.004854,0.501895,0.025000,0.012500,0.500000,0.000000,0.000000,0.973534,1.017423
Blobs_F2C2_S150,0.008591,0.002415,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.004167,0.014842,...,0.511246,0.011182,0.501551,0.016667,0.008333,0.500000,0.000000,0.000000,0.961642,1.017957
Blobs_F2C2_S200,0.008600,0.002343,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.003125,0.012792,...,0.509395,0.010250,0.500931,0.012500,0.006250,0.500000,0.000000,0.000000,0.956189,1.017602
Blobs_F2C3_S100,0.004046,0.001370,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.009376,0.016809,...,0.510933,0.002781,0.490811,0.037503,0.018751,0.500000,0.000171,0.000474,0.971605,1.690905
Blobs_F2C3_S150,0.004323,0.001300,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.006250,0.013097,...,0.510992,0.007291,0.493324,0.025000,0.012500,0.500000,0.000000,0.000000,0.962416,1.685753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Haberman_S150,0.778610,0.526454,0.478999,0.881667,0.873333,0.049134,0.231667,0.2475,0.147083,0.343089,...,0.882101,0.443768,0.663996,0.025000,0.025000,1.000000,0.163359,0.357664,0.267311,2.277940
Wine_S80,0.132954,0.029516,0.000005,0.098437,0.000000,0.000060,0.001563,0.0000,0.023438,0.342321,...,0.913542,0.563211,0.766817,0.203125,0.146875,0.723077,0.006349,0.017425,0.590979,4.218631
Wine_S100,0.169042,0.033700,0.000017,0.217500,0.000000,0.000030,0.003750,0.0000,0.028750,0.341432,...,0.935854,0.634435,0.800864,0.162500,0.125000,0.769231,0.007226,0.019802,0.570901,4.462597
Wine_S120,0.174444,0.035494,0.000024,0.211458,0.000000,0.000096,0.004167,0.0000,0.030729,0.347073,...,0.924518,0.595590,0.788415,0.135417,0.104167,0.769231,0.007841,0.021468,0.544426,4.768309


### 7.1 Merge Labels with Metrics

Join the Task-A best encoding circuit label with the complexity metric matrix to form the synthesis DataFrame — one row per dataset, 24 metric columns plus one label column.

In [ ]:
df["Best_Kernel"] = df_best["Best_Kernel"]
df

,f1,f1v,f2,f3,f4,l1,l2,l3,n1,n2,...,clsCoef,hubs,t2,t3,t4,c1,c2,kolmogorov,intrinsic,Best_Kernel
Blobs_F2C2_S100,0.007097,0.002049,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.006250,0.016410,...,0.004854,0.501895,0.025000,0.012500,0.500000,0.000000,0.000000,0.973534,1.017423,HZY_CZ
Blobs_F2C2_S150,0.008591,0.002415,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.004167,0.014842,...,0.011182,0.501551,0.016667,0.008333,0.500000,0.000000,0.000000,0.961642,1.017957,HZY_CZ
Blobs_F2C2_S200,0.008600,0.002343,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.003125,0.012792,...,0.010250,0.500931,0.012500,0.006250,0.500000,0.000000,0.000000,0.956189,1.017602,HZY_CZ
Blobs_F2C3_S100,0.004046,0.001370,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.009376,0.016809,...,0.002781,0.490811,0.037503,0.018751,0.500000,0.000171,0.000474,0.971605,1.690905,HighDim
Blobs_F2C3_S150,0.004323,0.001300,0.000000,0.000000,0.000000,0.000000,0.000000,0.0000,0.006250,0.013097,...,0.007291,0.493324,0.025000,0.012500,0.500000,0.000000,0.000000,0.962416,1.685753,HighDim
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Haberman_S150,0.778610,0.526454,0.478999,0.881667,0.873333,0.049134,0.231667,0.2475,0.147083,0.343089,...,0.443768,0.663996,0.025000,0.025000,1.000000,0.163359,0.357664,0.267311,2.277940,ZFeatureMap
Wine_S80,0.132954,0.029516,0.000005,0.098437,0.000000,0.000060,0.001563,0.0000,0.023438,0.342321,...,0.563211,0.766817,0.203125,0.146875,0.723077,0.006349,0.017425,0.590979,4.218631,HardwareEfficientRx
Wine_S100,0.169042,0.033700,0.000017,0.217500,0.000000,0.000030,0.003750,0.0000,0.028750,0.341432,...,0.634435,0.800864,0.162500,0.125000,0.769231,0.007226,0.019802,0.570901,4.462597,HighDim
Wine_S120,0.174444,0.035494,0.000024,0.211458,0.000000,0.000096,0.004167,0.0000,0.030729,0.347073,...,0.595590,0.788415,0.135417,0.104167,0.769231,0.007841,0.021468,0.544426,4.768309,HardwareEfficientRx


## 8. Save Outputs

Save three CSV files consumed by the downstream Majority Voting and LOOCV notebooks:
- `taska_labels.csv` — single-best labels
- `taskb_labels.csv` — expanded tolerance-based labels
- `synthesis_df.csv` — complexity metrics + Task-A labels (meta-dataset)

In [ ]:
df_best.to_csv("results/taska_labels.csv", index=True)
print("Saved: results/taska_labels.csv")

df_tied.to_csv("results/taskb_labels.csv", index=True)
print("Saved: results/taskb_labels.csv")

df.to_csv("results/synthesis_df.csv", index=True)
print("Saved: results/synthesis_df.csv")

Saved: results/taska_labels.csv
Saved: results/taskb_labels.csv
Saved: results/synthesis_df.csv
